In [1]:
import pandas as pd # Python library for data analysis and data frame
import numpy as np # Numerical Python library for linear algebra and computations
pd.set_option('display.max_columns', None) # code to display all columns

# Visualisation libraries
import matplotlib.pyplot as plt
import seaborn as sns

# libraries for text processing
import nltk
from nltk.stem.snowball import SnowballStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# to display images
from skimage import io

# to save the required files
import pickle

import warnings
warnings.filterwarnings('ignore') # To prevent kernel from showing any warning

In [2]:
df = pd.read_csv('../input/millions-of-movies/movies.csv')

Understand The Data 🧠

In [3]:
df.head()

,id,title,genres,original_language,overview,popularity,production_companies,release_date,budget,revenue,runtime,status,tagline,vote_average,vote_count,credits,keywords,poster_path,backdrop_path,recommendations
0,615656,Meg 2: The Trench,Action-Science Fiction-Horror,en,An exploratory dive into the deepest depths of...,8763.998,Apelles Entertainment-Warner Bros. Pictures-di...,2023-08-02,129000000.0,352056482.0,116.0,Released,Back for seconds.,7.079,1365.0,Jason Statham-Wu Jing-Shuya Sophia Cai-Sergio ...,based on novel or book-sequel-kaiju,/4m1Au3YkjqsxF8iwQy0fPYSxE0h.jpg,/qlxy8yo5bcgUw2KAmmojUKp4rHd.jpg,1006462-298618-569094-1061181-346698-1076487-6...
1,758323,The Pope's Exorcist,Horror-Mystery-Thriller,en,Father Gabriele Amorth Chief Exorcist of the V...,5953.227,Screen Gems-2.0 Entertainment-Jesus & Mary-Wor...,2023-04-05,18000000.0,65675816.0,103.0,Released,Inspired by the actual files of Father Gabriel...,7.433,545.0,Russell Crowe-Daniel Zovatto-Alex Essoe-Franco...,spain-rome italy-vatican-pope-pig-possession-c...,/9JBEPLTPSm0d1mbEcLxULjJq9Eh.jpg,/hiHGRbyTcbZoLsYYkO4QiCLYe34.jpg,713704-296271-502356-1076605-1084225-1008005-9...
2,667538,Transformers: Rise of the Beasts,Action-Adventure-Science Fiction,en,When a new threat capable of destroying the en...,5409.104,Skydance-Paramount-di Bonaventura Pictures-Bay...,2023-06-06,200000000.0,407045464.0,127.0,Released,Unite or fall.,7.340,1007.0,Anthony Ramos-Dominique Fishback-Luna Lauren V...,peru-alien-end of the world-based on cartoon-b...,/gPbM0MK8CP8A174rmUwGsADNYKD.jpg,/woJbg7ZqidhpvqFGGMRhWQNoxwa.jpg,496450-569094-298618-385687-877100-598331-4628...
3,693134,Dune: Part Two,Science Fiction-Adventure,en,Follow the mythic journey of Paul Atreides as ...,4742.163,Legendary Pictures,2024-02-27,190000000.0,683813734.0,167.0,Released,Long live the fighters.,8.300,2770.0,Timothée Chalamet-Zendaya-Rebecca Ferguson-Jav...,epic-based on novel or book-fight-sandstorm-sa...,/czembW0Rk1Ke7lCJGahbOhdCuhV.jpg,/xOMo8BRK7PfcJv9JCnx7s5hj0PX.jpg,438631-763215-792307-1011985-467244-634492-359...
4,640146,Ant-Man and the Wasp: Quantumania,Action-Adventure-Science Fiction,en,Super-Hero partners Scott Lang and Hope van Dy...,4425.387,Marvel Studios-Kevin Feige Productions,2023-02-15,200000000.0,475766228.0,125.0,Released,Witness the beginning of a new dynasty.,6.507,2811.0,Paul Rudd-Evangeline Lilly-Jonathan Majors-Kat...,hero-ant-sequel-superhero-based on comic-famil...,/qnqGbB22YJ7dSs4o6M7exTpNxPz.jpg,/m8JTwHFwX7I7JY5fPe4SjqejWag.jpg,823999-676841-868759-734048-267805-965839-1033...


In [4]:
df.shape

(722342, 20)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 722342 entries, 0 to 722341
Data columns (total 20 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   id                    722342 non-null  int64  
 1   title                 722338 non-null  object 
 2   genres                511992 non-null  object 
 3   original_language     722342 non-null  object 
 4   overview              604082 non-null  object 
 5   popularity            722342 non-null  float64
 6   production_companies  337376 non-null  object 
 7   release_date          670736 non-null  object 
 8   budget                722342 non-null  float64
 9   revenue               722342 non-null  float64
 10  runtime               687998 non-null  float64
 11  status                722342 non-null  object 
 12  tagline               108448 non-null  object 
 13  vote_average          722342 non-null  float64
 14  vote_count            722342 non-null  float64
 15  

In [6]:
df.isnull().sum()

id                           0
title                        4
genres                  210350
original_language            0
overview                118260
popularity                   0
production_companies    384966
release_date             51606
budget                       0
revenue                      0
runtime                  34344
status                       0
tagline                 613894
vote_average                 0
vote_count                   0
credits                 224730
keywords                511736
poster_path             184542
backdrop_path           499177
recommendations         686497
dtype: int64

In [7]:
df.duplicated().sum()

1

In [8]:
df.drop_duplicates(inplace=True)

Let's check if there are any movies with same title

In [9]:
df['title'].duplicated().sum()

147017

In [10]:
df[['title','release_date']].duplicated().sum()

62187

In [11]:
df.drop_duplicates(subset=['title','release_date'], inplace=True)

In [12]:
df.shape

(660154, 20)

Now we have 6 lakh movies but most of the movies have 0 vote count. so we will consider only those movies which have at least more than 20 vote counts.

In [13]:
df1 = df[df.vote_count >= 20].reset_index()

In [14]:
df1 = df1[df1.budget != 0].reset_index()

In [15]:
df1.isnull().sum()

level_0                    0
index                      0
id                         0
title                      0
genres                     2
original_language          0
overview                  19
popularity                 0
production_companies     334
release_date               0
budget                     0
revenue                    0
runtime                    0
status                     0
tagline                 2449
vote_average               0
vote_count                 0
credits                   19
keywords                1065
poster_path                2
backdrop_path            221
recommendations         2712
dtype: int64

In [16]:
df1.shape

(12017, 22)

In [17]:
df1[df1['genres'].isnull()]

,level_0,index,id,title,genres,original_language,overview,popularity,production_companies,release_date,budget,revenue,runtime,status,tagline,vote_average,vote_count,credits,keywords,poster_path,backdrop_path,recommendations
11914,43783,100856,137955,Crowsnest,NaN,en,In late summer of 2011 five young friends on a...,1.958,NaN,2012-01-01,1200000.0,0.0,84.0,Released,NaN,5.0,29.0,Mittita Barber-Aslam Husain-Victor Zinck Jr.-C...,found footage,/g2swX9YGaIq1dpGr7gva8DGnWe4.jpg,NaN,NaN
11940,44078,105337,124697,Almeno tu nell'universo,NaN,it,NaN,1.863,RAI,2011-08-12,1500000.0,0.0,0.0,Released,NaN,5.9,30.0,Giulia Elettra Gorietti-Giuseppe Maggio-Chiara...,NaN,/qweryHSqu6t9ZGN3zt10Tzshfb.jpg,/u0WXTe9b9tQo6Tjyio0tvXW2Iu3.jpg,56903-22971-37950-77877-272693-21943-60049-151...


In [18]:
index = df1[(df1['genres'].isnull()) & (df1['overview'].isnull())].index
df1.drop(index, inplace=True)
df1.isnull().sum()

level_0                    0
index                      0
id                         0
title                      0
genres                     1
original_language          0
overview                  18
popularity                 0
production_companies     334
release_date               0
budget                     0
revenue                    0
runtime                    0
status                     0
tagline                 2448
vote_average               0
vote_count                 0
credits                   19
keywords                1064
poster_path                2
backdrop_path            221
recommendations         2712
dtype: int64

In [19]:
df2 = df1.drop(columns = ['tagline','poster_path','backdrop_path','recommendations'],axis = 1)
df2.isnull().sum()

level_0                    0
index                      0
id                         0
title                      0
genres                     1
original_language          0
overview                  18
popularity                 0
production_companies     334
release_date               0
budget                     0
revenue                    0
runtime                    0
status                     0
vote_average               0
vote_count                 0
credits                   19
keywords                1064
dtype: int64

In [20]:
df2 = df2.fillna("Unknown")
df2.isnull().sum()

level_0                 0
index                   0
id                      0
title                   0
genres                  0
original_language       0
overview                0
popularity              0
production_companies    0
release_date            0
budget                  0
revenue                 0
runtime                 0
status                  0
vote_average            0
vote_count              0
credits                 0
keywords                0
dtype: int64

In [21]:
df2['status'].value_counts()

Released    12016
Name: status, dtype: int64

EDA

In [22]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 12016 entries, 0 to 12016
Data columns (total 18 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   level_0               12016 non-null  int64  
 1   index                 12016 non-null  int64  
 2   id                    12016 non-null  int64  
 3   title                 12016 non-null  object 
 4   genres                12016 non-null  object 
 5   original_language     12016 non-null  object 
 6   overview              12016 non-null  object 
 7   popularity            12016 non-null  float64
 8   production_companies  12016 non-null  object 
 9   release_date          12016 non-null  object 
 10  budget                12016 non-null  float64
 11  revenue               12016 non-null  float64
 12  runtime               12016 non-null  float64
 13  status                12016 non-null  object 
 14  vote_average          12016 non-null  float64
 15  vote_count         

In [23]:
categorical_columns = df2.select_dtypes(include=['object']).columns
numerical_columns = df2.select_dtypes(include=['number']).columns

print("Categorical Columns:", categorical_columns)
print("Numerical Columns:", numerical_columns)


Categorical Columns: Index(['title', 'genres', 'original_language', 'overview',
       'production_companies', 'release_date', 'status', 'credits',
       'keywords'],
      dtype='object')
Numerical Columns: Index(['level_0', 'index', 'id', 'popularity', 'budget', 'revenue', 'runtime',
       'vote_average', 'vote_count'],
      dtype='object')


In [24]:
df2[df2['runtime'] == 0]

,level_0,index,id,title,genres,original_language,overview,popularity,production_companies,release_date,budget,revenue,runtime,status,vote_average,vote_count,credits,keywords
2349,3647,4275,611014,Foxter and Max,Adventure-Fantasy-Family,uk,The twelve-year-old schoolboy Max escapes from...,24.809,Pronto Film,2019-09-19,1522860.0,0.0,0.0,Released,6.7,56.0,Bohdan Kozii-Vitaliia Turchyn-Daria Polunina-T...,technology-nanotechnology-teen movie-dog-robot...
6781,12900,15698,774456,Garcia & Garcia,Comedy,es,Two men with the same name endure the funny co...,10.331,Blogmedia,2021-08-27,3404274.0,1163538.0,0.0,Released,5.5,42.0,José Mota-Pepe Viyuela-Eva Ugarte-Martita de G...,airport-identity swap-assumed name
7402,14679,17791,398702,La Niña De La Mina,Thriller-Horror-Mystery,es,Arter a tourist disappears inside a mine in Gu...,9.405,Leow Films,2016-06-24,1200000.0,0.0,0.0,Released,6.0,26.0,Fernanda Sasse-Gerardo Taracena-Regina Blandón...,horror-mina
8758,19798,24073,412795,Maligno,Horror-Mystery,es,Unknown,7.442,Dolby,2016-07-14,150000.0,400000.0,0.0,Released,6.0,20.0,Sofía Rocha-Fiorella Pennano-Gino Pesaressi-Sy...,Unknown
10051,27026,33762,48941,Vajont,Drama,it,On October 9th 1963 at 10:39 pm 260 million cu...,5.466,Canal+-Les Productions Bagheera-RAI-S.D.P. Films,2001-10-19,12350000.0,0.0,0.0,Released,6.5,85.0,Michel Serrault-Daniel Auteuil-Laura Morante-J...,Unknown
10347,28988,36857,696826,Il regno,Comedy,it,The king of a secret modern-time Medieval king...,5.023,Fandango-RAI-MiBAC,2020-06-26,800000.0,0.0,0.0,Released,5.5,53.0,Stefano Fresi-Max Tortora-Silvia D'Amico-Fotin...,kingdom
10578,30844,40131,147686,Mes héros,Comedy,fr,Unknown,4.638,Eskwad-Josy Films-Pathé-Malec Productions,2012-12-10,9780000.0,0.0,0.0,Released,6.1,52.0,Josiane Balasko-Gérard Jugnot-Clovis Cornillac...,Unknown
10662,31543,41482,44148,Amar a Morir,Drama-Romance,es,Unknown,4.496,Laguna Films,2009-01-23,2500000.0,0.0,0.0,Released,7.1,42.0,Martina García-José María de Tavira-Alberto Es...,Unknown
11342,37206,55474,92117,Anderson Silva: Like Water,Documentary,pt,Anderson Silva is the deadliest man on the pla...,3.415,Unknown,2011-10-06,325.0,0.0,0.0,Released,6.3,23.0,Anderson Silva-Jose Aldo-Junior dos Santos,fighter-mixed martial arts
11833,42599,83951,698583,Chi ha incastrato Babbo Natale?,Comedy,it,Unknown,2.308,Bartlebyfilm-Buonaluna-Vision Distribution,2021-12-16,5000000.0,0.0,0.0,Released,5.8,22.0,Alessandro Siani-Christian De Sica-Sara Ciocca...,christmas
